**Fraud_Synthetic_Data**

In [36]:
import numpy as np
import pandas as pd
import random
import os
import time

In [37]:
np.random.seed(42)
random.seed(42)


In [38]:
print('Step 1 complete: imports loaded successfully')


Step 1 complete: imports loaded successfully


In [39]:
# STEP 2: Constants -- change these numbers to adjust the dataset
N_TOTAL   = 500000         
FRAUD_RATE = 0.05          
N_FRAUD   = int(N_TOTAL * FRAUD_RATE)  
N_HONEST  = N_TOTAL - N_FRAUD   
print(f'Total users : {N_TOTAL:,}')
print(f'Fraud users : {N_FRAUD:,}  ({FRAUD_RATE*100:.0f}%)')
print(f'N_Honest:{N_HONEST:,}')

# Cities -- 15 major Indian cities
CITIES = [
    'Mumbai', 'Delhi', 'Bengaluru', 'Hyderabad', 'Pune',
    'Chennai', 'Kolkata', 'Ahmedabad', 'Jaipur', 'Surat',
    'Lucknow', 'Kanpur', 'Nagpur', 'Indore', 'Bhopal'
]
CATEGORIES = [
    'Electronics', 'Fashion', 'Beauty & Personal Care',
    'Home & Kitchen', 'Books', 'Sports & Outdoor',
    'Grocery', 'Toys & Baby', 'Automotive'
]
# Fraud users target high-value categories more
FRAUD_CATEGORY_WEIGHTS  = [35, 28, 12, 10, 3, 5, 2, 3, 2]
# Honest users spread returns more evenly
HONEST_CATEGORY_WEIGHTS = [18, 22, 15, 18, 8, 7, 5, 4, 3]
 
print('Step 2 complete: constants defined')


Total users : 500,000
Fraud users : 25,000  (5%)
N_Honest:475,000
Step 2 complete: constants defined


**STEP 3 — the fraud user generator function**

In [40]:
#STEP 3 — Write the fraud user generator function
def generate_fraud_user(user_id):
    '''Creates one fraudulent user profile.
    Every value is based on documented fraud patterns.'''
    return_rate = np.random.beta(8, 4) * 100
    # -- RETURN RATE --
    # Beta(8,4): average = 8/(8+4) = 67%. Fraudsters return MOST of their orders.
    damage_claim = np.random.beta(7, 3) * 100
    # -- DAMAGE CLAIM RATE --
    # Beta(7,3): average = 7/(7+3) = 70%. Fraudsters almost always claim damage.
    # Damage is the easiest excuse -- cannot be verified remotely.
    
    # -- DAYS TO RETURN (BIMODAL PATTERN) --
    # 60% return QUICKLY (1-4 days): used the item, returning fast
    # 40% return at WARRANTY CLIFF (5-8 days): maxed out the return window
    # This bimodal pattern is unique to fraudsters -- honest users don't do this
    if random.random() < 0.60:
        days_to_return = random.randint(1, 4)   # quick use and return
    else:
        days_to_return = random.randint(5, 8)   # warranty cliff exploitation
 
    # -- ACCOUNT AGE --
    # Fraud accounts are new (7 to 200 days)
    # Fraudsters create new accounts when old ones get flagged
    account_age = random.randint(7, 200)
 
    # -- ADDRESSES --
    # 2 to 8 addresses: fraud rings use multiple delivery points
    addresses = random.randint(2, 8)
 
    # -- ORDER VOLUME --
    # 2 to 15: fraudsters order in bursts (order lots, return all, disappear)
    orders_90d = random.randint(2, 15)
 
    # -- HIGH VALUE RETURNS --
    # 1 to 6: fraudsters specifically target expensive items
    hv_returns = random.randint(1, 6)
 
    # -- NEGATIVE REVIEWS --
    # 0 to 4: posted strategically as evidence/cover for return claims
    neg_reviews = random.randint(0, 4)
 
    # -- SUPPORT TICKETS --
    # 2 to 9: fraudsters escalate repeatedly to pressure approvals
    support_tix = random.randint(2, 9)
 
    # -- PAYMENT METHOD CHANGES --
    # 2 to 7: fraud rings rotate payment methods to avoid detection
    pay_changes = random.randint(2, 7)
 
    # -- WEEKEND ORDER PERCENTAGE --
    # Beta(4,3): average = 4/7 = 57%. Some fraudsters target weekends.
    weekend_pct = np.random.beta(4, 3) * 100
 
    # -- CATEGORY CONCENTRATION --
    # Beta(6,2): average = 6/8 = 0.75. Fraudsters are specialists.
    # Score close to 1.0 means ALL returns from same category.
    cat_conc = np.random.beta(6, 2)
 
    # -- TOP RETURN CATEGORY --
    # Weighted: Electronics(35%) and Fashion(28%) most common for fraud
    top_cat = random.choices(CATEGORIES, weights=FRAUD_CATEGORY_WEIGHTS)[0]
 
    # min(value, 99.0) caps at 99% to avoid impossible 100% values
    return {
        'user_id'                       : f'U{user_id:07d}',
        'city'                          : random.choice(CITIES),
        'account_age_days'              : account_age,
        'orders_last_90d'               : orders_90d,
        'return_rate_pct'               : round(min(return_rate, 99.0), 1),
        'damage_claim_pct'              : round(min(damage_claim, 99.0), 1),
        'avg_days_to_return'            : days_to_return,
        'unique_addresses_used'         : addresses,
        'high_value_return_count'       : hv_returns,
        'negative_reviews_after_return' : neg_reviews,
        'support_tickets_filed'         : support_tix,
        'payment_method_changes'        : pay_changes,
        'weekend_order_pct'             : round(weekend_pct, 1),
        'category_concentration_score'  : round(cat_conc, 3),
        'top_return_category'           : top_cat,
        'is_fraud'                      : 1
    }
 
test_fraud = generate_fraud_user(1)
print('Test fraud user:')
for key, value in test_fraud.items():
    print(f'  {key:35s} = {value}')
print('Step 3 complete: fraud generator working')


Test fraud user:
  user_id                             = U0000001
  city                                = Jaipur
  account_age_days                    = 196
  orders_last_90d                     = 5
  return_rate_pct                     = 72.8
  damage_claim_pct                    = 72.5
  avg_days_to_return                  = 5
  unique_addresses_used               = 4
  high_value_return_count             = 2
  negative_reviews_after_return       = 1
  support_tickets_filed               = 3
  payment_method_changes              = 7
  weekend_order_pct                   = 64.8
  category_concentration_score        = 0.652
  top_return_category                 = Beauty & Personal Care
  is_fraud                            = 1
Step 3 complete: fraud generator working


**STEP 4 — the honest user generator function**

In [41]:
def  generate_honest_user(user_id):
    '''Creates one honest user profile.
    Key principle: distributions OVERLAP with fraud at the edges.
    Some honest users have high returns (bad luck). That is realistic.
    If there is zero overlap, any simple rule would catch all fraud'''
    # -- RETURN RATE --
    # Beta(2,9): average = 2/11 = 18%. Low return rates.left side bell
    # But Beta distribution has a tail -- some honest users reach 40-50%
    # This overlap with fraud users is INTENTIONAL
    return_rate = np.random.beta(2, 9) * 100
 
    # -- DAMAGE CLAIM RATE --
    # Beta(1,6): average = 1/7 = 14%. Honest users rarely claim damage.
    damage_claim = np.random.beta(1, 6) * 100
 
    # -- DAYS TO RETURN --
    # Uniform 1-18: random -- genuine returns have no specific timing pattern
    days_to_return = random.randint(1, 18)
 
    # -- ACCOUNT AGE --
    # 60 to 1800 days: established accounts, up to 5 years
    account_age = random.randint(60, 1800)
 
    # -- ADDRESSES --
    # Mostly 1-2. Rarely 3 (moved home, gift deliveries)
    # random.random() gives a number between 0 and 1
    # if it is less than 0.08 (8% chance), give 3 addresses
    addresses = random.randint(1, 2)
    if random.random() < 0.08:
        addresses = 3
 
    # -- ORDER VOLUME --
    # 2 to 40: steady regular shoppers
    orders_90d = random.randint(2, 40)
 
    # -- HIGH VALUE RETURNS --
    # weighted: 70% chance of 0, 25% chance of 1, 5% chance of 2
    hv_returns = random.choices([0, 1, 2], weights=[70, 25, 5])[0]
 
    # -- NEGATIVE REVIEWS --
    # 75% have zero, 20% have one, 5% have two
    neg_reviews = random.choices([0, 1, 2], weights=[75, 20, 5])[0]
 
    # -- SUPPORT TICKETS --
    # 55% have zero, 30% have one, 12% have two, 3% have three
    support_tix = random.choices([0, 1, 2, 3], weights=[55, 30, 12, 3])[0]
 
    # -- PAYMENT METHOD CHANGES --
    # 65% unchanged, 28% one change (new card), 7% two changes
    pay_changes = random.choices([0, 1, 2], weights=[65, 28, 7])[0]
 
    # -- WEEKEND ORDER PERCENTAGE --
    # Beta(3,4): average = 3/7 = 43%. Natural weekend shopping pattern.
    weekend_pct = np.random.beta(3, 4) * 100
 
    # -- CATEGORY CONCENTRATION --
    # Beta(2,5): average = 2/7 = 0.28. Honest users spread returns around.
    cat_conc = np.random.beta(2, 5)
 
    # -- TOP RETURN CATEGORY --
    # More evenly spread for honest users
    top_cat = random.choices(CATEGORIES, weights=HONEST_CATEGORY_WEIGHTS)[0]
 
    return {
        'user_id'                       : f'U{user_id:07d}',
        'city'                          : random.choice(CITIES),
        'account_age_days'              : account_age,
        'orders_last_90d'               : orders_90d,
        'return_rate_pct'               : round(min(return_rate, 99.0), 1),
        'damage_claim_pct'              : round(min(damage_claim, 99.0), 1),
        'avg_days_to_return'            : days_to_return,
        'unique_addresses_used'         : addresses,
        'high_value_return_count'       : hv_returns,
        'negative_reviews_after_return' : neg_reviews,
        'support_tickets_filed'         : support_tix,
        'payment_method_changes'        : pay_changes,
        'weekend_order_pct'             : round(weekend_pct, 1),
        'category_concentration_score'  : round(cat_conc, 3),
        'top_return_category'           : top_cat,
        'is_fraud'                      : 0
    }
 
# Quick test
test_honest = generate_honest_user(99999)
print('Test honest user:')
for key, value in test_honest.items():
    print(f'  {key:35s} = {value}')
print('Step 4 complete: honest generator working')


Test honest user:
  user_id                             = U0099999
  city                                = Jaipur
  account_age_days                    = 1269
  orders_last_90d                     = 7
  return_rate_pct                     = 32.4
  damage_claim_pct                    = 16.7
  avg_days_to_return                  = 3
  unique_addresses_used               = 3
  high_value_return_count             = 0
  negative_reviews_after_return       = 0
  support_tickets_filed               = 0
  payment_method_changes              = 0
  weekend_order_pct                   = 38.6
  category_concentration_score        = 0.342
  top_return_category                 = Home & Kitchen
  is_fraud                            = 0
Step 4 complete: honest generator working


**STEP 5 — The Vectorized (Fast) Main Generation Function**

In [55]:
# ============================================================
# STEP 5: Vectorized main generation function (FAST METHOD)
# ============================================================
 
def generate_all_users_vectorized():
    '''
    Generated all 500,000 users using numpy vectorized operations.
    '''
    start_time = time.time()
    print(f'Generating {N_TOTAL:,} users...')
    print(f'  Fraud users:  {N_FRAUD:,}')
    print(f'  Honest users: {N_HONEST:,}')
    print()
 
    # ================================================================
    # GENERATE ALL FRAUD USERS AT ONCE (vectorized)
    # ================================================================
    print('  Generating fraud users (vectorized)...')
 
    # numpy generates ALL N_FRAUD values in one call -- very fast
    fraud_return_rates   = np.random.beta(8, 4, N_FRAUD) * 100
    fraud_damage_claims  = np.random.beta(7, 3, N_FRAUD) * 100
    fraud_account_ages   = np.random.randint(7, 201, N_FRAUD)
    fraud_orders         = np.random.randint(2, 16, N_FRAUD)
    fraud_addresses      = np.random.randint(2, 9, N_FRAUD)
    fraud_hv_returns     = np.random.randint(1, 7, N_FRAUD)
    fraud_neg_reviews    = np.random.randint(0, 5, N_FRAUD)
    fraud_support        = np.random.randint(2, 10, N_FRAUD)
    fraud_pay_changes    = np.random.randint(2, 8, N_FRAUD)
    fraud_weekend        = np.random.beta(4, 3, N_FRAUD) * 100
    fraud_cat_conc       = np.random.beta(6, 2, N_FRAUD)
 
    # Bimodal days_to_return: 60% quick, 40% cliff
    # Generate all quick returns, all cliff returns, then choose
    quick_returns = np.random.randint(1, 5, N_FRAUD)   # 1-4 days
    cliff_returns = np.random.randint(5, 9, N_FRAUD)   # 5-8 days
    is_quick = np.random.random(N_FRAUD) < 0.60         # 60% are quick
    fraud_days_to_return = np.where(is_quick, quick_returns, cliff_returns)
 
    # Weighted category choices (still needs a loop -- unavoidable for choices)
    fraud_categories = [
        random.choices(CATEGORIES, weights=FRAUD_CATEGORY_WEIGHTS)[0]
        for _ in range(N_FRAUD)
    ]
    fraud_cities = [random.choice(CITIES) for _ in range(N_FRAUD)]
 
    # Build fraud DataFrame directly from arrays -- much faster than dict append
    fraud_df = pd.DataFrame({
        'user_id'                       : [f'U{i+1:07d}' for i in range(N_FRAUD)],
        'city'                          : fraud_cities,
        'account_age_days'              : fraud_account_ages,
        'orders_last_90d'               : fraud_orders,
        'return_rate_pct'               : np.clip(fraud_return_rates, 0, 99.0).round(1),
        'damage_claim_pct'              : np.clip(fraud_damage_claims, 0, 99.0).round(1),
        'avg_days_to_return'            : fraud_days_to_return,
        'unique_addresses_used'         : fraud_addresses,
        'high_value_return_count'       : fraud_hv_returns,
        'negative_reviews_after_return' : fraud_neg_reviews,
        'support_tickets_filed'         : fraud_support,
        'payment_method_changes'        : fraud_pay_changes,
        'weekend_order_pct'             : fraud_weekend.round(1),
        'category_concentration_score'  : fraud_cat_conc.round(3),
        'top_return_category'           : fraud_categories,
        'is_fraud'                      : 1
    })
    print(f'  Fraud users generated: {len(fraud_df):,}')
 
    # ================================================================
    # GENERATE ALL HONEST USERS AT ONCE (vectorized)
    # ================================================================
    print('  Generating honest users (vectorized)...')
 
    honest_return_rates  = np.random.beta(2, 9, N_HONEST) * 100
    honest_damage_claims = np.random.beta(1, 6, N_HONEST) * 100
    honest_account_ages  = np.random.randint(60, 1801, N_HONEST)
    honest_orders        = np.random.randint(2, 41, N_HONEST)
    honest_weekend       = np.random.beta(3, 4, N_HONEST) * 100
    honest_cat_conc      = np.random.beta(2, 5, N_HONEST)
    honest_days          = np.random.randint(1, 19, N_HONEST)
 
    # Addresses: mostly 1-2, 8% chance of 3
    honest_addresses = np.random.randint(1, 3, N_HONEST)
    three_address_mask = np.random.random(N_HONEST) < 0.08
    honest_addresses[three_address_mask] = 3
 
    # Weighted counts using numpy for speed
    honest_hv_returns = np.random.choice([0,1,2], N_HONEST, p=[0.70, 0.25, 0.05])
    honest_neg_reviews= np.random.choice([0,1,2], N_HONEST, p=[0.75, 0.20, 0.05])
    honest_support    = np.random.choice([0,1,2,3], N_HONEST, p=[0.55, 0.30, 0.12, 0.03])
    honest_pay_changes= np.random.choice([0,1,2], N_HONEST, p=[0.65, 0.28, 0.07])
 
    # Category and city still need list comprehension
    honest_categories = [
        random.choices(CATEGORIES, weights=HONEST_CATEGORY_WEIGHTS)[0]
        for _ in range(N_HONEST)
    ]
    honest_cities = [random.choice(CITIES) for _ in range(N_HONEST)]
 
    honest_df = pd.DataFrame({
        'user_id'                       : [f'U{N_FRAUD+i+1:07d}' for i in range(N_HONEST)],
        'city'                          : honest_cities,
        'account_age_days'              : honest_account_ages,
        'orders_last_90d'               : honest_orders,
        'return_rate_pct'               : np.clip(honest_return_rates, 0, 99.0).round(1),
        'damage_claim_pct'              : np.clip(honest_damage_claims, 0, 99.0).round(1),
        'avg_days_to_return'            : honest_days,
        'unique_addresses_used'         : honest_addresses,
        'high_value_return_count'       : honest_hv_returns,
        'negative_reviews_after_return' : honest_neg_reviews,
        'support_tickets_filed'         : honest_support,
        'payment_method_changes'        : honest_pay_changes,
        'weekend_order_pct'             : honest_weekend.round(1),
        'category_concentration_score'  : honest_cat_conc.round(3),
        'top_return_category'           : honest_categories,
        'is_fraud'                      : 0
    })
    print(f'  Honest users generated: {len(honest_df):,}')
 
    # ================================================================
    # COMBINE, SHUFFLE, AND SAVE
    # ================================================================
    print('  Combining and shuffling...')
    df = pd.concat([fraud_df, honest_df], ignore_index=True)
 
    # MUST shuffle: without this, all fraud users are rows 1-25000
    # A train-test split on unshuffled data would have ALL fraud in train
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)
 
    elapsed = time.time() - start_time
    print(f'  Generation complete in {elapsed:.1f} seconds')
 
    return df
 
print('Step 5 complete: main function defined')

if __name__ == '__main__':
    df = generate_all_users_vectorized()
  


Step 5 complete: main function defined
Generating 500,000 users...
  Fraud users:  25,000
  Honest users: 475,000

  Generating fraud users (vectorized)...
  Fraud users generated: 25,000
  Generating honest users (vectorized)...
  Honest users generated: 475,000
  Combining and shuffling...
  Generation complete in 2.3 seconds


In [43]:
def validate_and_save(df):
    '''
    Runs 8 validation checks on the generated data.
    Saves to CSV.
    '''
    print()
    print('=' * 60)
    print('VALIDATION CHECKS')
    print('=' * 60)
 
    # Check 1: Shape
    print(f'CHECK 1 -- Shape: {df.shape}')
    assert df.shape[0] == N_TOTAL, f'FAIL: Expected {N_TOTAL} rows, got {df.shape[0]}'
    assert df.shape[1] == 16,      f'FAIL: Expected 16 columns, got {df.shape[1]}'
    print('  PASS: 500,000 rows x 16 columns')
 
    # Check 2: Fraud rate
    actual_fraud_rate = df['is_fraud'].mean() * 100
    print(f'CHECK 2 -- Fraud rate: {actual_fraud_rate:.1f}%')
    assert abs(actual_fraud_rate - 5.0) < 0.1, 'FAIL: Fraud rate not 5%'
    print('  PASS: Fraud rate is 5.0%')
 
    # Check 3: No missing values
    missing = df.isnull().sum().sum()
    print(f'CHECK 3 -- Missing values: {missing}')
    assert missing == 0, 'FAIL: Missing values found'
    print('  PASS: Zero missing values')
 
    # Check 4: Return rate bounds
    assert df['return_rate_pct'].min() >= 0, 'FAIL: Negative return rate'
    assert df['return_rate_pct'].max() <= 99, 'FAIL: Return rate above 99'
    print(f'CHECK 4 -- Return rate range: {df["return_rate_pct"].min():.1f}% to {df["return_rate_pct"].max():.1f}%  PASS')
 
    # Check 5: Fraud users have higher return rates
    fraud_rr  = df[df['is_fraud']==1]['return_rate_pct'].mean()
    honest_rr = df[df['is_fraud']==0]['return_rate_pct'].mean()
    assert fraud_rr > honest_rr * 2, 'FAIL: Fraud return rate not much higher than honest'
    print(f'CHECK 5 -- Return rate: Fraud avg={fraud_rr:.1f}%  Honest avg={honest_rr:.1f}%  Ratio={fraud_rr/honest_rr:.1f}x  PASS')
 
    # Check 6: No duplicate user IDs
    duplicates = df['user_id'].duplicated().sum()
    assert duplicates == 0, f'FAIL: {duplicates} duplicate user IDs'
    print(f'CHECK 6 -- Duplicate user IDs: {duplicates}  PASS')
 
    # Check 7: All cities valid
    invalid_cities = df[~df['city'].isin(CITIES)]
    assert len(invalid_cities) == 0, 'FAIL: Invalid city names found'
    print(f'CHECK 7 -- All cities valid  PASS')
 
    # Check 8: Category concentration in range
    assert df['category_concentration_score'].min() >= 0, 'FAIL: Concentration below 0'
    assert df['category_concentration_score'].max() <= 1, 'FAIL: Concentration above 1'
    print(f'CHECK 8 -- Category concentration range valid  PASS')
 
    print()
    print('ALL 8 VALIDATION CHECKS PASSED!')
    print()
 
    # ================================================================
    # PRINT SUMMARY STATISTICS (memorise these for interviews)
    # ================================================================
    print('=' * 60)
    print('DATASET SUMMARY -- MEMORISE THESE FOR INTERVIEWS')
    print('=' * 60)
 
    fraud  = df[df['is_fraud']==1]
    honest = df[df['is_fraud']==0]
 
    print(f'Total users     : {len(df):,}')
    print(f'Fraud users     : {len(fraud):,} ({len(fraud)/len(df)*100:.1f}%)')
    print(f'Honest users    : {len(honest):,} ({len(honest)/len(df)*100:.1f}%)')
    print(f'Class imbalance : {len(honest)//len(fraud)}:1 (honest to fraud)')
    print(f'Total columns   : {len(df.columns)}')
    print()
 
    cols = ['return_rate_pct','damage_claim_pct','account_age_days',
            'unique_addresses_used','support_tickets_filed']
    for col in cols:
        f_avg = fraud[col].mean()
        h_avg = honest[col].mean()
        ratio = f_avg / h_avg if h_avg > 0 else 0
        print(f'{col:38s}: Fraud={f_avg:6.1f}  Honest={h_avg:6.1f}  Ratio={ratio:.1f}x')
 
    # ================================================================
if __name__ == "__main__":
    validate_and_save(df)


VALIDATION CHECKS
CHECK 1 -- Shape: (500000, 16)
  PASS: 500,000 rows x 16 columns
CHECK 2 -- Fraud rate: 5.0%
  PASS: Fraud rate is 5.0%
CHECK 3 -- Missing values: 0
  PASS: Zero missing values
CHECK 4 -- Return rate range: 0.0% to 98.7%  PASS
CHECK 5 -- Return rate: Fraud avg=66.6%  Honest avg=18.2%  Ratio=3.7x  PASS
CHECK 6 -- Duplicate user IDs: 0  PASS
CHECK 7 -- All cities valid  PASS
CHECK 8 -- Category concentration range valid  PASS

ALL 8 VALIDATION CHECKS PASSED!

DATASET SUMMARY -- MEMORISE THESE FOR INTERVIEWS
Total users     : 500,000
Fraud users     : 25,000 (5.0%)
Honest users    : 475,000 (95.0%)
Class imbalance : 19:1 (honest to fraud)
Total columns   : 16

return_rate_pct                       : Fraud=  66.6  Honest=  18.2  Ratio=3.7x
damage_claim_pct                      : Fraud=  70.1  Honest=  14.3  Ratio=4.9x
account_age_days                      : Fraud= 103.0  Honest= 929.7  Ratio=0.1x
unique_addresses_used                 : Fraud=   5.0  Honest=   1.6  Ratio=

**SAVE CSV File**

In [54]:
df.to_csv("Synthetic_fraud_Data.csv", index=False)

In [53]:
print("File saved successfully!")

File saved successfully!
